In [11]:
import torch

if torch.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

device

device(type='cuda')

In [12]:
import torch
import torch.nn as nn
class SimpleMathCNN(nn.Module):
    def __init__(self, num_classes = 10):
        super().__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding = 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride = 2),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=64 * 7 * 7, out_features=128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.fc(x)
        return x




In [13]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

In [14]:
def train(model, optimizer, loss_fn, metric, train_loader, valid_loader, n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            y_pred = model(x_batch)
            loss = loss_fn(y_pred,y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history



In [16]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28,28)),
    transforms.ToTensor(),
    transforms.Normalize(mean = (0.5,),  std = (0.5,))
])

dataset_path = 'D:/Projekty/kalkulator/model/data'
full_dataset = datasets.ImageFolder(root = dataset_path, transform=transform)

total_size = len(full_dataset)
train_size = int(0.8 * total_size)
valid_size = total_size - train_size

generator = torch.Generator()
train_dataset, valid_dataset = random_split(
    full_dataset,
    [train_size, valid_size],
    generator=generator
)

BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

num_classes = len(full_dataset.classes)
print(f"Liczba obrazków: {total_size} (Train: {train_size}, Valid: {valid_size})")
print(f"Liczba klas: {num_classes}")



Liczba obrazków: 10071 (Train: 8056, Valid: 2015)
Liczba klas: 1


In [19]:
from torch import optim as optim
import torchmetrics

num_classes = 19


metric = torchmetrics.F1Score(
    task="multiclass",
    num_classes=num_classes,
    average="macro"
).to(device)

model = SimpleMathCNN(num_classes=num_classes).to(device)

n_epochs = 200
loss_fn = nn.CrossEntropyLoss()
lr = 0.001
optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

ValueError: Expected argument `num_classes` to be an integer larger than 1, but got 1

In [18]:
train(model, optimizer, loss_fn, metric, train_loader, valid_loader, n_epochs)

NameError: name 'metric' is not defined